In [5]:
# CÉLULA 1: Instalação das dependências
!pip install highspy pandas numpy -q

In [6]:
# CÉLULA 2: Local Branching TRADICIONAL (Fischetti & Lodi, 2002)
import os
import time
import numpy as np
import highspy

def build_ulsr_model(datafile):
    """Lê o arquivo de dados e constrói o modelo MIP base no HiGHS."""
    with open(datafile, 'r') as file:
        linhas = [a.strip() for a in file.readlines() if a.strip()]

    N = int(linhas[0])
    FR_val, F_val  = float(linhas[1]), float(linhas[2])
    HR_val, H_val  = float(linhas[3]), float(linhas[4])

    D = np.array([float(v) for v in linhas[5].split()], dtype=float)
    R = np.array([float(v) for v in linhas[6].split()], dtype=float)

    P, PR = np.zeros(N), np.zeros(N)
    F, FR = np.full(N, F_val), np.full(N, FR_val)
    H, HR = np.full(N, H_val), np.full(N, HR_val)

    D_suffix = np.cumsum(D[::-1])[::-1]
    R_prefix = np.cumsum(R)

    h = highspy.Highs()
    inf = highspy.kHighsInf

    idx_x  = list(range(0, N))
    idx_s  = list(range(N, 2*N))
    idx_y  = list(range(2*N, 3*N))
    idx_xr = list(range(3*N, 4*N))
    idx_sr = list(range(4*N, 5*N))
    idx_yr = list(range(5*N, 6*N))

    for i in range(N): h.addCol(P[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(H[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(F[i], 0.0, 1.0, 0, [], [])
    for i in range(N): h.addCol(PR[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(HR[i], 0.0, inf, 0, [], [])
    for i in range(N): h.addCol(FR[i], 0.0, 1.0, 0, [], [])

    for i in range(N):
        h.changeColIntegrality(idx_y[i], highspy.HighsVarType.kInteger)
        h.changeColIntegrality(idx_yr[i], highspy.HighsVarType.kInteger)

    h.addRow(D[0], D[0], 3, [idx_x[0], idx_xr[0], idx_s[0]], [1.0, 1.0, -1.0])
    for i in range(1, N):
        h.addRow(D[i], D[i], 4, [idx_s[i-1], idx_x[i], idx_xr[i], idx_s[i]], [1.0, 1.0, 1.0, -1.0])

    h.addRow(-R[0], -R[0], 2, [idx_xr[0], idx_sr[0]], [-1.0, -1.0])
    for i in range(1, N):
        h.addRow(-R[i], -R[i], 3, [idx_sr[i-1], idx_xr[i], idx_sr[i]], [1.0, -1.0, -1.0])

    for i in range(N):
        big_m = float(D_suffix[i])
        h.addRow(-inf, 0.0, 2, [idx_x[i], idx_y[i]], [1.0, -big_m])

    for i in range(N):
        cap_max = min(float(D_suffix[i]), float(R_prefix[i]))
        h.addRow(-inf, 0.0, 2, [idx_xr[i], idx_yr[i]], [1.0, -cap_max])

    h.addRow(0.0, 0.0, 1, [idx_s[N-1]], [1.0])

    binary_indices = idx_y + idx_yr
    return h, binary_indices

def ulsr_local_branching(datafile, time_limit=300.0, sub_time_limit=10.0, k_radius=10, debug=False):
    """
    Algoritmo Local Branching TRADICIONAL (Fischetti & Lodi, 2002).
    - Raio k fixo de vizinhança.
    - Resolução sequencial de subproblemas delimitados por distância de Hamming.
    """
    start_time = time.time()

    # 1. Fase Inicial Tradicional: Obtenção de Solução Incumbente em 10s
    h_init, binary_indices = build_ulsr_model(datafile)
    h_init.setOptionValue("output_flag", debug)
    h_init.setOptionValue("presolve", "on")
    h_init.setOptionValue("time_limit", float(min(10.0, time_limit)))
    h_init.run()

    sol_init = h_init.getSolution()
    info_init = h_init.getInfo()

    if not sol_init.value_valid:
        return {
            'INSTANCIA': os.path.basename(datafile),
            'BOUND': None,
            'OBJVAL': None,
            'GAP': None,
            'TIME': time.time() - start_time,
            'OTIMALIDADE': 'Infeaso'
        }

    best_obj = info_init.objective_function_value
    best_bound = getattr(info_init, "mip_dual_bound", None)
    incumbent_sol = np.round([sol_init.col_value[idx] for idx in binary_indices])

    # 2. Loop de Local Branching Tradicional
    while (time.time() - start_time) < time_limit:
        remaining_time = time_limit - (time.time() - start_time)
        if remaining_time <= 0.5:
            break

        h, bin_idxs = build_ulsr_model(datafile)
        h.setOptionValue("output_flag", debug)
        h.setOptionValue("presolve", "on")
        h.setOptionValue("time_limit", float(min(sub_time_limit, remaining_time)))

        # Restrição da Vizinhança Local: Hamming(z, z_bar) <= k
        cols, coeffs = [], []
        ones_count = 0
        for idx, val in zip(bin_idxs, incumbent_sol):
            if val >= 0.5:
                cols.append(idx)
                coeffs.append(-1.0)
                ones_count += 1
            else:
                cols.append(idx)
                coeffs.append(1.0)

        rhs_upper = float(k_radius - ones_count)
        h.addRow(-highspy.kHighsInf, rhs_upper, len(cols), cols, coeffs)

        h.run()
        info = h.getInfo()
        status = h.getModelStatus()

        # Se encontrar uma solução estritamente melhor na vizinhança
        if status in [highspy.HighsModelStatus.kOptimal, highspy.HighsModelStatus.kTimeLimit]:
            sol = h.getSolution()
            if sol.value_valid and info.objective_function_value < (best_obj - 1e-4):
                best_obj = info.objective_function_value
                incumbent_sol = np.round([sol.col_value[idx] for idx in bin_idxs])
                continue

        # No Local Branching clássico, quando a vizinhança atual não gera melhorias, a busca encerra
        break

    exec_time = time.time() - start_time
    gap = (best_obj - best_bound) / abs(best_obj) if (best_bound is not None and best_obj is not None and best_obj != 0) else None

    return {
        'INSTANCIA': os.path.basename(datafile),
        'BOUND': best_bound,
        'OBJVAL': best_obj,
        'GAP': gap,
        'TIME': exec_time,
        'OTIMALIDADE': 'Local Search'
    }

In [7]:
# CÉLULA 3: Execução do Experimento 1 (Local Branching Clássico)
import os
import re
import glob
import pandas as pd
from IPython.display import display

def chave_natural(caminho):
    nome = os.path.basename(caminho)
    partes = re.split(r'(\d+)', nome)
    return [int(p) if p.isdigit() else p for p in partes]

pasta          = "/content/drive/MyDrive/Colab Notebooks/sifaleras"
extensao       = "*.txt"
time_limit     = 300.0  # Limite global de 5 minutos por instância
sub_time_limit = 10.0   # Limite por subproblema (padrão literatura)
k_radius       = 10     # ~10% das 104 variáveis binárias

inicio = 1
fim    = 108  # Ajuste para o número total das suas instâncias

arquivos = sorted(glob.glob(os.path.join(pasta, extensao)), key=chave_natural)
arquivos = arquivos[inicio-1 : fim]

print(f"--- EXPERIMENTO 1: Local Branching Tradicional ---")
print(f"Instâncias: {len(arquivos)} | Tempo Total Max={time_limit}s | Sub-limite={sub_time_limit}s | Raio k={k_radius}\n")

resultados = []
for arq in arquivos:
    print(f"A processar: {os.path.basename(arq)}...", end=" ", flush=True)
    res = ulsr_local_branching(
        arq,
        time_limit=time_limit,
        sub_time_limit=sub_time_limit,
        k_radius=k_radius,
        debug=False
    )
    print("Concluído!")
    resultados.append(res)

df = pd.DataFrame(resultados)

tabela = pd.DataFrame({
    'INSTANCIA':   df['INSTANCIA'],
    'BOUND':       df['BOUND'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'OBJVAL':      df['OBJVAL'].map(lambda v: f"{v:.2f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'GAP':         df['GAP'].map(lambda v: f"{v*100:.3f}%" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'TIME':        df['TIME'].map(lambda v: f"{v:.4f}" if pd.notnull(v) and isinstance(v, (int, float)) else "N/A"),
    'OTIMALIDADE': df['OTIMALIDADE'],
})

display(tabela)

--- EXPERIMENTO 1: Local Branching Tradicional ---
Instâncias: 108 | Tempo Total Max=300.0s | Sub-limite=10.0s | Raio k=10

A processar: 52_1.txt... Concluído!
A processar: 52_2.txt... Concluído!
A processar: 52_3.txt... Concluído!
A processar: 52_4.txt... Concluído!
A processar: 52_5.txt... Concluído!
A processar: 52_6.txt... Concluído!
A processar: 52_7.txt... Concluído!
A processar: 52_8.txt... Concluído!
A processar: 52_9.txt... Concluído!
A processar: 52_10.txt... Concluído!
A processar: 52_11.txt... Concluído!
A processar: 52_12.txt... Concluído!
A processar: 52_13.txt... Concluído!
A processar: 52_14.txt... Concluído!
A processar: 52_15.txt... Concluído!
A processar: 52_16.txt... Concluído!
A processar: 52_17.txt... Concluído!
A processar: 52_18.txt... Concluído!
A processar: 52_19.txt... Concluído!
A processar: 52_20.txt... Concluído!
A processar: 52_21.txt... Concluído!
A processar: 52_22.txt... Concluído!
A processar: 52_23.txt... Concluído!
A processar: 52_24.txt... Concluíd

,INSTANCIA,BOUND,OBJVAL,GAP,TIME,OTIMALIDADE
0,52_1.txt,8586.16,8736.80,1.724%,20.0418,Local Search
1,52_2.txt,8588.17,8819.40,2.622%,30.0657,Local Search
2,52_3.txt,8426.99,8541.60,1.342%,20.0521,Local Search
3,52_4.txt,8773.44,8952.60,2.001%,30.0794,Local Search
4,52_5.txt,9427.83,9767.00,3.473%,30.0542,Local Search
...,...,...,...,...,...,...
103,52_104.txt,27678.87,37634.50,26.453%,20.0338,Local Search
104,52_105.txt,35838.80,39001.40,8.109%,20.0481,Local Search
105,52_106.txt,33370.48,41270.60,19.142%,20.0463,Local Search
106,52_107.txt,36860.74,39120.40,5.776%,20.0411,Local Search


In [8]:

import os
import pandas as pd
from IPython.display import FileLink

# Define o caminho completo e o nome do arquivo para salvar a tabela no Google Drive
filename_csv = '/content/drive/MyDrive/Colab Notebooks/resultado_Local_Branching_Highs.csv'
filename_xlsx = '/content/drive/MyDrive/Colab Notebooks/resultado_Local_Branching_Highs.xlsx'

# Salva o DataFrame 'tabela' como um arquivo CSV
tabela.to_csv(filename_csv, index=False, encoding='utf-8')

# Salva o DataFrame 'tabela' como um arquivo XLSX
tabela.to_excel(filename_xlsx, index=False)

# Adiciona uma linha para mostrar o caminho completo dos arquivos salvos
print(f"A tabela foi salva como '{os.path.basename(filename_csv)}' no caminho: {filename_csv}")
print(f"A tabela também foi salva como '{os.path.basename(filename_xlsx)}' no caminho: {filename_xlsx}")

A tabela foi salva como 'resultado_Local_Branching_Highs.csv' no caminho: /content/drive/MyDrive/Colab Notebooks/resultado_Local_Branching_Highs.csv
A tabela também foi salva como 'resultado_Local_Branching_Highs.xlsx' no caminho: /content/drive/MyDrive/Colab Notebooks/resultado_Local_Branching_Highs.xlsx
